In [635]:
import numpy as np

from numba import njit


from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

import optuna

import time

# Init Reservoir

In [636]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [637]:
u_val = 1
u_dist = 200
u_dataset = (1 - t // u_dist % 2) * 2 * u_val - u_val
u_dataset = u_dataset[transient_steps_chaos:]

# Init Funcs

In [638]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scattergl(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scattergl3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [639]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scattergl(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scattergl(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [640]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [641]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [642]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]
    
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

# Calc Init

In [643]:
@njit(fastmath=True, cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))

    idx_a, idx_b = connections_list[:, 0], connections_list[:, 1]

    disp_reshaped = disp.reshape(
        num_nodes, dims
    )

    pos_a = initial_pos[idx_a] + disp_reshaped[idx_a]
    pos_b = initial_pos[idx_b] + disp_reshaped[idx_b]

    r_vecs = pos_b - pos_a
    current_lens = np.sqrt(np.sum(r_vecs**2, axis=1))

    force_magnitudes = k_vals * (current_lens - rest_lens)

    safe_lens = np.where(current_lens < 1e-13, 1.0, current_lens)
    unit_dirs = r_vecs / safe_lens.reshape(-1, 1)

    forces[idx_a] += force_magnitudes[:, np.newaxis] * unit_dirs
    forces[idx_b] -= force_magnitudes[:, np.newaxis] * unit_dirs

    return forces.reshape(-1)

In [644]:
@njit(fastmath=True, cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    init_vecs = (
        initial_pos[connections_list[:, 0]] - initial_pos[connections_list[:, 1]]
    )
    rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )
    F_spring *= mask

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )
        F_spring *= mask

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Chain with Constant Var

In [341]:
N = 100

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [342]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = np.ones(num_nodes) * .2
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = np.ones(num_nodes) * .1
c_diag = np.repeat(node_c, dims)

In [ ]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

k_vals = np.ones(src_nodes.shape[0]) * .7
connections_list = np.column_stack((src_nodes, dst_nodes))

In [344]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[:, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[:, :col_indices.shape[0]]

In [345]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.005,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

In [346]:
X = np.column_stack((displacement, velocity))

X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

Y_data = u_dataset[transient_steps_reservoir + tau_steps:]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)

Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9280 0.2684


In [347]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [340]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Chain with Random Var

In [ ]:
N = 100

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [428]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

rng = np.random.default_rng(42)

node_m = rng.uniform(0.1, 0.3, size=num_nodes)
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = rng.uniform(0.05, 0.3, size=num_nodes)
c_diag = np.repeat(node_c, dims)

In [358]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 1, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [359]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[:, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[:, :col_indices.shape[0]]

In [360]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.005,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

In [363]:
X = np.column_stack((displacement, velocity))

X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

Y_data = u_dataset[transient_steps_reservoir + tau_steps:]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)

Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9164 0.2891


In [ ]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [ ]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# 20 Chain with Optuna with Constant Vars

In [429]:
N = 20

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [430]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [431]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

connections_list = np.column_stack((src_nodes, dst_nodes))

In [432]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[:, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[:, : col_indices.shape[0]]

In [509]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha):
    node_m = np.ones(num_nodes) * m_val
    m_diag = np.repeat(node_m, dims)
    m_inv_diag = 1.0 / m_diag
    node_c = np.ones(num_nodes) * c_val
    c_diag = np.repeat(node_c, dims)
    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=steps + transient_steps_reservoir + tau_steps,
        dt=0.005,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=[-1],
    )

    X = np.column_stack((displacement, velocity))
    positions = nodes_pos + displacement[-1].reshape(-1, dims)

    positions = nodes_pos.flatten() + displacement
    gaps = np.diff(positions, axis=1)
    if np.any(gaps < 0.05):
        return (), (-1.0, 1e9), ()

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps:]

    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [510]:
_, (r_2, mse), _ = bayesian_trial(0.2, 0.1, 0.7, 0.4, .1)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9279 0.2685


In [519]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse

study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [520]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #60
  Values: [0.9732658973966051, 0.16350566535565356]
  Params: {'mass': 0.03832750256066554, 'damping': 0.02197418106108189, 'stiffness': 2.4676364803513846, 'input_force': 0.48255288102935706, 'ridge_alpha': 0.001542582697363626}


In [521]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"]
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9733 0.1635


In [522]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [467]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# 100 Chain with Optuna with Constant Vars

In [523]:
N = 100

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [524]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

In [525]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

connections_list = np.column_stack((src_nodes, dst_nodes))

In [526]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[:, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[:, : col_indices.shape[0]]

In [527]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha):
    node_m = np.ones(num_nodes) * m_val
    m_diag = np.repeat(node_m, dims)
    m_inv_diag = 1.0 / m_diag
    node_c = np.ones(num_nodes) * c_val
    c_diag = np.repeat(node_c, dims)
    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=steps + transient_steps_reservoir + tau_steps,
        dt=0.005,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=[-1],
    )

    X = np.column_stack((displacement, velocity))
    positions = nodes_pos + displacement[-1].reshape(-1, dims)

    positions = nodes_pos.flatten() + displacement
    gaps = np.diff(positions, axis=1)
    if np.any(gaps < 0.05):
        return (), (-1.0, 1e9), ()

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps:]

    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [530]:
_, (r_2, mse), _ = bayesian_trial(0.2, 0.1, 0.7, 0.4, .1)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9280 0.2684


In [531]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
    )

    return r_2, mse

study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [532]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #7
  Values: [0.9856752899001978, 0.11968588095428026]
  Params: {'mass': 0.020077283546146265, 'damping': 0.03539990146481307, 'stiffness': 5.380730773680139, 'input_force': 0.9526342074324479, 'ridge_alpha': 0.043819024440707184}


In [533]:
params = study.best_trials[0].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"]
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9857 0.1197


In [534]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [ ]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# N Chain with Optuna with Constant Vars

In [543]:
def bayesian_trial(m_val, c_val, k_val, input_force, ridge_alpha, N):
    x = np.arange(N)
    nodes_pos = x.reshape(-1, 1)

    tau_steps = 1
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]
    matrix_size = num_nodes * dims

    node_ids = np.arange(x.size)
    src_nodes = node_ids[:-1]
    dst_nodes = node_ids[1:]
    connections_list = np.column_stack((src_nodes, dst_nodes))

    target_nodes = np.array([0])
    force_data = u_dataset.reshape(-1, dims)
    U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
    col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
    ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
    U[:, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[:, : col_indices.shape[0]]

    node_m = np.ones(num_nodes) * m_val
    m_diag = np.repeat(node_m, dims)
    m_inv_diag = 1.0 / m_diag
    node_c = np.ones(num_nodes) * c_val
    c_diag = np.repeat(node_c, dims)
    k_vals = np.ones(src_nodes.shape[0]) * k_val

    displacement, velocity = run_simulation(
        steps=steps + transient_steps_reservoir + tau_steps,
        dt=0.005,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * input_force,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=[-1],
    )

    X = np.column_stack((displacement, velocity))
    positions = nodes_pos + displacement[-1].reshape(-1, dims)

    positions = nodes_pos.flatten() + displacement
    gaps = np.diff(positions, axis=1)
    if np.any(gaps < 0.05):
        return (), (-1.0, 1e9), ()

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps:]

    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return (Y_test, Y_pred), (r_2, mse), (displacement, velocity)

In [544]:
_, (r_2, mse), _ = bayesian_trial(0.2, 0.1, 0.7, 0.4, .1, 10)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9216 0.2799


In [547]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    start_time = time.perf_counter()
    _, (r_2, mse), _ = bayesian_trial(
        trial.suggest_float("mass", 0.01, 10, log=True),
        trial.suggest_float("damping", 0.01, 10, log=True),
        trial.suggest_float("stiffness", 0.01, 10, log=True),
        trial.suggest_float("input_force", 1e-3, 100.0, log=True),
        trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
        trial.suggest_int("N", 3, 1000)
    )
    duration = time.perf_counter() - start_time
    if r_2 == -1.0:
        duration = 10.0

    return r_2, mse, duration

study = optuna.create_study(directions=["maximize", "minimize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [549]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #24
  Values: [0.9782680958532887, 0.14741744858296577, 6.158093457983341]
  Params: {'mass': 0.011634430893781173, 'damping': 1.1513135928186853, 'stiffness': 5.41476107291508, 'input_force': 4.032672673014153, 'ridge_alpha': 0.007237819713250291, 'N': 263}
Trial #35
  Values: [0.9632955111170148, 0.1915841561376754, 0.4005493330187164]
  Params: {'mass': 0.052784098715047405, 'damping': 5.530278013844606, 'stiffness': 1.5956967650885243, 'input_force': 0.6261565173800238, 'ridge_alpha': 8.054345686906165, 'N': 65}


In [566]:
params = study.best_trials[1].params
(Y_test, Y_pred), (r_2, mse), (displacement, velocity) = bayesian_trial(
    params["mass"],
    params["damping"],
    params["stiffness"],
    params["input_force"],
    params["ridge_alpha"],
    params["N"]
)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9633 0.1916


In [562]:
nodes_pos = np.arange(params["N"]).reshape(-1, 1)

node_ids = np.arange(params["N"])
src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]
connections_list = np.column_stack((src_nodes, dst_nodes))

spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [565]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

# Free Time Chain

In [758]:
N = 20

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [759]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = np.ones(num_nodes) * 0.2
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = np.ones(num_nodes) * 0.1
c_diag = np.repeat(node_c, dims)

In [760]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

k_vals = np.ones(src_nodes.shape[0]) * 0.7
connections_list = np.column_stack((src_nodes, dst_nodes))

In [761]:
free_steps = 1
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

In [762]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [680]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=0.01,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

In [681]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [682]:
X = np.column_stack((displacement, velocity))

X_sampled = X[::free_steps]
X_delayed = X_sampled[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

Y_data = u_dataset[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)

Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9362 0.2526


In [713]:
free_step_range = np.arange(1, 30)
results = np.zeros((free_step_range.shape[0], 4))

for i, free_steps in enumerate(free_step_range):
    start_time = time.perf_counter()

    total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

    target_nodes = np.array([0])
    force_data = u_dataset.reshape(-1, dims)

    U = np.zeros((total_steps_with_free, matrix_size))
    col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
    ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
    U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
        :, : col_indices.shape[0]
    ]

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=0.005,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * 0.4,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=[-1],
    )

    X = np.column_stack((displacement, velocity))
    X_sampled = X[::free_steps]
    X_delayed = X_sampled[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps :]
    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    duration = time.perf_counter() - start_time

    results[i, 0] = free_steps
    results[i, 1] = r2_score(Y_test, Y_pred)
    results[i, 2] = root_mean_squared_error(Y_test, Y_pred)
    results[i, 3] = duration

In [723]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    template="plotly_dark",
)

fig.update_yaxes(title_text="R² Score", secondary_y=False)
fig.update_yaxes(title_text="MSE", secondary_y=True)

fig.show()

In [724]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]
time_vals = results[:, 3]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=time_vals,
        name="Time (s)",
        line=dict(color="gold", dash="dot", width=3),
        yaxis="y3",
    )
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    yaxis=dict(title="R² Score"),
    yaxis2=dict(title="MSE", overlaying="y", side="right"),
    yaxis3=dict(
        title="Time (s)", overlaying="y", side="right", position=.85
    ),
    margin=dict(r=100),
    template="plotly_dark",
)

fig.show()

In [763]:
free_steps = 5

total_steps_with_free = free_steps * (
    steps + transient_steps_reservoir + tau_steps
)

target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=0.005,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

X = np.column_stack((displacement, velocity))
X_sampled = X[::free_steps]
X_delayed = X_sampled[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]
Y_data = u_dataset[transient_steps_reservoir + tau_steps :]
X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

In [764]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9606 0.1985


In [ ]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()

: 

# Free Time Chain Using All Data

In [730]:
N = 20

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [731]:
tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = np.ones(num_nodes) * 0.2
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = np.ones(num_nodes) * 0.1
c_diag = np.repeat(node_c, dims)

In [732]:
node_ids = np.arange(x.size)

src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]

k_vals = np.ones(src_nodes.shape[0]) * 0.7
connections_list = np.column_stack((src_nodes, dst_nodes))

In [738]:
free_steps = 3
total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

In [739]:
target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

In [740]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=0.01,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

In [741]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
    vel=velocity,
).show()

In [750]:
X = np.column_stack((displacement, velocity))

X_delayed = X[:-tau_steps * free_steps]
X_data = X_delayed[transient_steps_reservoir * free_steps :]

Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)

rest_test_steps = test_steps * free_steps

X_train, X_test = (
    X_data[:-rest_test_steps],
    X_data[-rest_test_steps:],
)

Y_train, Y_test = (
    Y_data[:-rest_test_steps],
    Y_data[-rest_test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9739 0.1614


In [751]:
free_step_range = np.arange(1, 30)
results = np.zeros((free_step_range.shape[0], 4))

for i, free_steps in enumerate(free_step_range):
    start_time = time.perf_counter()

    total_steps_with_free = free_steps * (steps + transient_steps_reservoir + tau_steps)

    target_nodes = np.array([0])
    force_data = u_dataset.reshape(-1, dims)

    U = np.zeros((total_steps_with_free, matrix_size))
    col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
    ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
    U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
        :, : col_indices.shape[0]
    ]

    displacement, velocity = run_simulation(
        steps=total_steps_with_free,
        dt=0.005,
        m_inv_diag=m_inv_diag,
        c_diag=c_diag,
        U=U * 0.4,
        initial_pos=nodes_pos,
        connections_list=connections_list,
        k_vals=k_vals,
        wall_nodes=[-1],
    )

    X = np.column_stack((displacement, velocity))
    X_delayed = X[:-tau_steps * free_steps]
    X_data = X_delayed[transient_steps_reservoir * free_steps :]
    Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)
    rest_test_steps = test_steps * free_steps
    X_train, X_test = (
        X_data[:-rest_test_steps],
        X_data[-rest_test_steps:],
    )
    Y_train, Y_test = (
        Y_data[:-rest_test_steps],
        Y_data[-rest_test_steps:],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    duration = time.perf_counter() - start_time

    results[i, 0] = free_steps
    results[i, 1] = r2_score(Y_test, Y_pred)
    results[i, 2] = root_mean_squared_error(Y_test, Y_pred)
    results[i, 3] = duration

In [753]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    template="plotly_dark",
)

fig.update_yaxes(title_text="R² Score", secondary_y=False)
fig.update_yaxes(title_text="MSE", secondary_y=True)

fig.show()

In [754]:
free_steps = results[:, 0]
r2_vals = results[:, 1]
mse_vals = results[:, 2]
time_vals = results[:, 3]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=free_steps, y=r2_vals, name="R² Score", line=dict(color="cyan", width=3)
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=mse_vals,
        name="MSE",
        line=dict(color="firebrick", dash="dash", width=3),
    ),
    secondary_y=True,
)

fig.add_trace(
    go.Scatter(
        x=free_steps,
        y=time_vals,
        name="Time (s)",
        line=dict(color="gold", dash="dot", width=3),
        yaxis="y3",
    )
)

fig.update_layout(
    title="Reservoir Performance vs. Relaxation Steps",
    xaxis_title="Relaxation Steps",
    yaxis=dict(title="R² Score"),
    yaxis2=dict(title="MSE", overlaying="y", side="right"),
    yaxis3=dict(title="Time (s)", overlaying="y", side="right", position=0.85),
    margin=dict(r=100),
    template="plotly_dark",
)

fig.show()

In [755]:
free_steps = 10

total_steps_with_free = free_steps * (
    steps + transient_steps_reservoir + tau_steps
)

target_nodes = np.array([0])
force_data = u_dataset.reshape(-1, dims)

U = np.zeros((total_steps_with_free, matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
ceil_repeats = (col_indices.shape[0] + dims - 1) // force_data.shape[1]
U[::free_steps, col_indices] = np.repeat(force_data, ceil_repeats, axis=1)[
    :, : col_indices.shape[0]
]

displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=0.005,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * 0.4,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[-1],
)

X = np.column_stack((displacement, velocity))
X_delayed = X[:-tau_steps * free_steps]
X_data = X_delayed[transient_steps_reservoir * free_steps :]
Y_data = u_dataset[transient_steps_reservoir + tau_steps :].repeat(free_steps)
rest_test_steps = test_steps * free_steps
X_train, X_test = (
    X_data[:-rest_test_steps],
    X_data[-rest_test_steps:],
)
Y_train, Y_test = (
    Y_data[:-rest_test_steps],
    Y_data[-rest_test_steps:],
)
model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

In [756]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9758 0.1554


In [757]:
weight_plot(model.coef_).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()